In [4]:
!pip install "vllm>=0.8.5" mcp ddgs smolagents markdownify

  Using cached vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl.metadata (18 kB)
  Using cached compressed_tensors-0.12.2-py3-none-any.whl.metadata (7.0 kB)
Using cached vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl (370.3 MB)
Using cached compressed_tensors-0.12.2-py3-none-any.whl (183 kB)


In [5]:
!nvidia-smi

Thu Nov 20 10:04:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import typing as tp
import subprocess

# Model setup

**Visit https://openrouter.ai/ to create an account, generate api key and search for models**

In [12]:
# PROVIDER = 'OpenRouter' # OpenRouter or vLLM
PROVIDER = 'vLLM' # OpenRouter or vLLM

In [13]:
from getpass import getpass
if PROVIDER == 'vLLM':
    KEY = "EMPTY"
    HOST = "127.0.0.1"
    PORT = "8999"
    MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    URL = f"http://{HOST}:{PORT}/v1"
else:
    MODEL = "x-ai/grok-4.1-fast"
    KEY = getpass("Your openrouter api key: ")
    URL = f"https://openrouter.ai/api/v1"


In [14]:
if PROVIDER == 'vLLM':
    vllm_server = subprocess.Popen([
        "python",
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--max_model_len", "16384",
        "--host", HOST,
        "--port", PORT,
        "--gpu_memory_utilization", "0.6",
        "--max_num_seqs", "16",
    ])
    #  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --max_model_len 16384 --host 127.0.0.1 --port 8999 --gpu_memory_utilization 0.6 --max_num_seqs 16 --temperature 0.9

# vllm_server.terminate() — остановить


In [17]:
from openai import OpenAI

client = OpenAI(
  base_url=URL,
  api_key=KEY,
)


response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi"}],
)

print(response.choices[0].message)


ChatCompletionMessage(content='Hello! How can I assist you today? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning=None, reasoning_content=None)


# How to write tools? How to use MCP?

## Tools as functions

In [18]:
from ddgs import DDGS

def duckduckgo_search(query: str, max_results: int = 5):
    """

    """
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results, )
        return list(results)


In [19]:
duckduckgo_search('yandex data school')

[{'title': 'School of data analysis',
  'href': 'https://dataschool.yandex.com/',
  'body': 'A joint educational initiative of the Yandex School of Data Analysis, JetBrains and the Computer Science club. CS teaches software development, modern computer science, and data analysis.'},
 {'title': 'Academic programs at the Yandex School of Data Analysis',
  'href': 'https://dataschool.yandex.com/education',
  'body': 'The Yandex School of Data Analysis provides a systematic education, combining practice with theoretical knowledge. Each subject starts with the basics and culminates at the frontiers of science.'},
 {'title': 'Online courses - School of data analysis',
  'href': 'https://dataschool.yandex.com/dataschool/online',
  'body': 'If you are just starting to get acquainted with data analysis, we recommend taking a free introductory course with Practicum by Yandex. In this course, you will gain understanding of what data analysis is, what processes it includes, and what an analyst doe

In [20]:
import re
import requests
from markdownify import markdownify
from requests.exceptions import RequestException
def get_webpage_content(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()

        # Convert the HTML content to Markdown
        markdown_content = markdownify(response.text).strip()
        # Remove multiple line breaks
        markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)

        return markdown_content

    except RequestException as e:
        return f"Error fetching the webpage: {str(e)}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

In [21]:
# Implement function to get top-5 results with full text
def websearch_full_text(query, top_k=1) -> dict[str, tp.Any]:
    # <YOUR CODE HERE>
    pass

In [22]:
# Test your code

## Run MCP Server

## Connect MCP client

In [30]:
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [31]:
class MCPClient:
    def __init__(self, mcps: list[str]):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.mcps = mcps
        self.tools = []

    async def connect_to_server(self, server_script_path: str):
        command = "python"
        server_params = StdioServerParameters(
            command=command,
            args=[server_script_path],
            env=None
        )

        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))
        await self.session.initialize()
        response = await self.session.list_tools()
        self.tools = response.tools

    def list_tools(self):
        print("\nConnected to server with tools:", [tool.name for tool in self.tools])
        return self.tools

    async def call_tool(self, tool_name, args):
        result = await self.session.call_tool(tool_name, args)
        return result



In [ ]:
client = MCPClient('')
await client.connect_to_server('./ysda_tools.py')
tools = client.list_tools()

In [ ]:
tools[0]

# Tool calls example

In [36]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_youtube_captions",
            "description": "Fetch YouTube captions for a given video ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "video_id": {"type": "string"},
                    "lang": {"type": "string", "default": "en"}
                },
                "required": ["video_id"]
            }
        }
    }
]


In [37]:
import inspect
import typing

PYTHON_TO_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

def create_tool_description(func):
    sig = inspect.signature(func)
    doc = inspect.getdoc(func) or ""
    params_schema = {"type": "object", "properties": {}, "required": []}

    for name, param in sig.parameters.items():
        annotation = param.annotation
        if annotation in PYTHON_TO_JSON:
            json_type = PYTHON_TO_JSON[annotation]
        else:
            json_type = "string"  # fallback

        entry = {"type": json_type}

        if param.default is not inspect.Parameter.empty:
            entry["default"] = param.default
        else:
            params_schema["required"].append(name)

        params_schema["properties"][name] = entry

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": doc,
            "parameters": params_schema,
        },
    }


In [38]:
# Convert python function to openai tool

# ReAct Agent from scratch

In [39]:
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[query], which searches the web for the query and returns url, title and small snippet for 5 relevant pages.
(2) Visit web-page[link], which returns content of the page provided.
(3) Finish[answer], which returns the answer and finishes the task.
"""


final_prompt = "YOUR PROMPT WITH FEW SHOTS"

In [40]:
def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
      model=MODEL,
      prompt=prompt,
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    )
    return response["choices"][0]["text"]

In [ ]:
def execute_action(action: str) -> str:
    pass


def call_react(question, prompt=final_prompt, to_print=True):
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        # Generate thought
        thought_action = <YOUR CODE HERE>
        try:
            # Extract generated action
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except Exception as e:
            # Error handling
        # Execute action
        ... = execute_action()
        # Create observation
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        # Add observation to history
    return r, info

# Multi-agent systems with smolagents

In [65]:
from smolagents import tool


@tool
def visit_webpage(link: str) -> str:
  """
  Visit web-page[link], which returns content of the page provided.
  Args:
    link: link to the webpage
  Returns:
    content of the webpage
  """
  return get_webpage_content(link)

In [66]:
from smolagents import (
    CodeAgent,
    OpenAIModel,
    ToolCallingAgent,
    WebSearchTool,
)


model = OpenAIModel(
    model_id=MODEL,
    api_base=URL,
    api_key=KEY,
)


web_agent = CodeAgent(
    tools=[WebSearchTool(), visit_webpage],
    model=model,
    max_steps=5,
    name="web_search_agent",
    description="Runs web searches for you.",
)

In [67]:
!nvidia-smi

Thu Nov 20 11:08:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             33W /   70W |   12920MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [68]:
manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
)

In [69]:
answer = manager_agent.run(
    "How does ReAct agent works? What metrics were reported by authors?"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How does ReAct agent works? What metrics were reported by authors?                                              │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3-4B-Instruct-2507 ─────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = web_search_agent(task="Explain how the ReAct agent works and what metrics were reported by      
  authors in research papers", additional_args={"max_results": 3})                                                 
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── New run - web_search_agent ───────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'web_search_agent'.                                                                │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Explain how the ReAct agent works and what metrics were reported by authors in research papers                  │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│ You have been provided with these additional arguments, that you can access directly using the keys as          │
│ variables:                                                                                                      │
│ {'max_results': 3}.                                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3-4B-Instruct-2507 ─────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 1: Duration 52.95 seconds| Input tokens: 2,200 | Output tokens: 104]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The ReAct agent combines reasoning and acting to solve complex tasks by generating a               
  chain-of-thought process and executing external tools. Metrics reported by authors include task success rate,    
  reasoning quality, tool use efficiency, and sample efficiency.")                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The ReAct agent combines reasoning and acting to solve complex tasks by generating a chain-of-thought
process and executing external tools. Metrics reported by authors include task success rate, reasoning quality, 
tool use efficiency, and sample efficiency.

[Step 2: Duration 5.50 seconds| Input tokens: 4,963 | Output tokens: 234]

In [70]:
answer

'The ReAct agent combines reasoning and acting to solve complex tasks by generating a chain-of-thought process and executing external tools. Metrics reported by authors include task success rate, reasoning quality, tool use efficiency, and sample efficiency.'

In [71]:
from IPython.display import display, Markdown, Latex
display(Markdown(answer))

The ReAct agent combines reasoning and acting to solve complex tasks by generating a chain-of-thought process and executing external tools. Metrics reported by authors include task success rate, reasoning quality, tool use efficiency, and sample efficiency.